## Retrieve Memories from Azure CosmosDB

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up the Environment Variables

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Define the Vector Embedding Generator Helper Function

In [ ]:
def generate_emebedding(text):

    response = azure_openai_client.embeddings.create(
        model = embedding_model_name,
        input = text
    )

    return response.data[0].embedding

### Create the Memory Retrieval Function

In [ ]:
def retrieve_memories(customer_id, query_text, top_k=3):

    query_embedding = generate_emebedding(
        query_text
    )

    query = f"""
        SELECT TOP {top_k}

            c.id,
            c.customer_id,
            c.content,
            c.memory_type,
            c.importance,
            c.source,

            VectorDistance(
                c.embedding,
                @query_embedding
            ) AS distance

        FROM c

        WHERE
            c.customer_id = @customer_id

        ORDER BY
            VectorDistance(
                c.embedding,
                @query_embedding
            )
    """

    parameters = [

            {
                "name": "@customer_id",
                "value": customer_id
            },

            {
                "name": "@query_embedding",
                "value": query_embedding
            }
    ]

    results = list(
        container.query_items(
            query = query,
            parameters = parameters,
            partition_key = customer_id
        )
    )


    return results

### Define the User Query

Queries to try out:

1. Customer 1001 — Laptop Troubleshooting

```python
customer_id = "CUST-1001"

user_query = """
I'm having laptop problems again.

Before we start, can you tell me what troubleshooting
I've already tried and how I prefer to be contacted?
"""
```

2. Customer 1002 — Bluetooth Connectivity

```python
customer_id = "CUST-1002"

user_query = """
I'm having Bluetooth connectivity problems again.

What Bluetooth troubleshooting have I already performed?
"""
```

3. Customer 1003 — Billing Issues

```python
customer_id = "CUST-1003"

user_query = """
Have I had any billing problems before?
"""
```

In [ ]:
customer_id = "CUST-1001"

user_query = """
I'm having laptop problems again.

Before we start, can you tell me what troubleshooting
I've already tried and how I prefer to be contacted?
"""

### Retrieve Memories

In [ ]:
memories = retrieve_memories(customer_id=customer_id, query_text=user_query)

for memory in memories:
    print(f"- [{memory['memory_type']}] "
            f"{memory['content']}")

### Send an API Call to the LLM

In [ ]:
assistant_response = azure_openai_client.chat.completions.create(
    model = chat_completions_model_name,
    messages=[
        {
            "role": "system",
            "content": """You are a customer support assistant.

            You may receive relevant long-term memories
            from previous interactions with the customer.

            Use these memories when they are relevant
            to the current request.

            Do not invent customer history.

            Do not claim the customer previously said
            something unless it appears in the provided
            memory.

            Avoid asking the customer to repeat
            troubleshooting that has already been
            completed."""
        },
        {
            "role": "user",
            "content": f"""User query: {user_query}
                           Retrieved long-term memories: {memories}"""
        }
    ]
)


print(assistant_response.choices[0].message.content)